In [ ]:

from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

HF_TOKEN='pcsk_5MC14P_E2YfmAtTNcM7jpqBPp5ojGyK1n2vYpBx1EiQr23bJxq5Ezb6NfBjqqayxPh3wp8'

# =========================
# CONFIG
# =========================

PDF_PATH = "/home/moin/Downloads/Sample-pdf.pdf"

EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

FAISS_INDEX_PATH = "faiss_index"


# =========================
# LOAD PDF
# =========================

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"Loaded Pages: {len(documents)}")


# =========================
# SPLIT DOCUMENTS
# =========================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

split_docs = text_splitter.split_documents(documents)

print(f"Chunks Created: {len(split_docs)}")

from pathlib import Path

from langchain_core.documents import Document


def merge_documents_by_file(
    documents: list[Document]
) -> list[Document]:

    merged_docs = {}

    for doc in documents:

        source = doc.metadata.get("source", "unknown")

        filename = Path(source).name

        if filename not in merged_docs:
            merged_docs[filename] = []

        merged_docs[filename].append(
            doc.page_content
        )

    final_documents = []

    for filename, contents in merged_docs.items():

        merged_text = "\n\n".join(contents)

        final_documents.append(
            Document(
                page_content=merged_text,
                metadata={
                    "filename": filename
                }
            )
        )

    return final_documents

docs=merge_documents_by_file(split_docs)
print(docs)


Loaded Pages: 10
Chunks Created: 12


[Document(metadata={'filename': 'Sample-pdf.pdf'}, page_content='Sample PDF Document\nRobert Maron\nGrzegorz Grudzi´nski\nFebruary 20, 1999\n\n2\n\nContents\n1 Template 5\n1.1 How to compile a .tex ﬁle to a.pdf ﬁle . . . . . . . . . . . . . 5\n1.1.1 Tools . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5\n1.1.2 How to use the tools . . . . . . . . . . . . . . . . . . . . 5\n1.2 How to write a document . . . . . . . . . . . . . . . . . . . . . . 6\n1.2.1 The main document . . . . . . . . . . . . . . . . . . . . . 6\n1.2.2 Chapters . . . . . . . . . . . . . . . . . . . . . . . . . . 6\n1.2.3 Spell-checking . . . . . . . . . . . . . . . . . . . . . . . 6\n1.3 L ATEX and pdfLATEX capabilities . . . . . . . . . . . . . . . . . . . 7\n1.3.1 Overview . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n1.3.2 L ATEX . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n1.3.3 pdfL ATEX . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n1.3.4 Examples . . . . . . . . . . . . .

In [2]:
split_docs=split_docs[:2]
print(split_docs)

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20200629161449', 'source': '/home/moin/Downloads/Sample-pdf.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}, page_content='Sample PDF Document\nRobert Maron\nGrzegorz Grudzi´nski\nFebruary 20, 1999'), Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20200629161449', 'source': '/home/moin/Downloads/Sample-pdf.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2'}, page_content='2')]


In [ ]:
from langchain_pinecone import PineconeEmbeddings
import os
os.environ['PINECONE_API_KEY']=HF_TOKEN


In [26]:
from langchain_core.documents import Document
docs=[Document(page_content='my name is moin'), Document(page_content='lion is the king of jungle')]

In [ ]:

# =========================
# EMBEDDINGS
# =========================
import os
os.environ['HF_TOKEN']=HF_TOKEN
embeddings = PineconeEmbeddings(model="multilingual-e5-large")

print("Embedding model loaded")


# =========================
# CREATE FAISS INDEX
# =========================

vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

vectorstore.save_local(FAISS_INDEX_PATH)

print("FAISS index created successfully")



# =========================
# LOAD INDEX
# =========================

loaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded")


# =========================
# SIMILARITY SEARCH
# =========================

query = "king of jungle"

results = loaded_vectorstore.similarity_search_with_relevance_scores(
    query=query,
    k=3
)

print("\nTop Results:\n")

# for index, result in enumerate(results, start=1):
#     print(f"Result {index}")
#     print("-" * 50)
#     print(result.page_content[:500])
#     print("\n")

Embedding model loaded
FAISS index created successfully
